In [1]:
## Set up

import os
import shutil
import sqlite3
import time

import pandas as pd
import matplotlib.pyplot as plt

SOURCE = r"C:\Users\shana\Downloads\DATA501\ride-knox-analysis\sql\ride_knox_v2.db" # PART A -- Built the database in SQL
WORK   = "ride_knox_work.db"

if os.path.exists(WORK):
    os.remove(WORK)
shutil.copy(SOURCE, WORK)

con = sqlite3.connect(WORK)
con.isolation_level = None     # BEGIN / COMMIT / ROLLBACK behave as written


def run(sql):
    """Execute ONE statement. Returns rows affected (-1 if it changes no rows)."""
    return con.execute(sql).rowcount


def try_run(sql):
    """For statements you EXPECT to be refused: prints the database's
    complaint instead of stopping the notebook."""
    try:
        n = con.execute(sql).rowcount
        return f"ACCEPTED ({n} rows) — is that what you expected?"
    except sqlite3.Error as e:
        return f"REFUSED -> {e}"


def timed(sql, params=(), repeat=5):
    """Run a query a few times; return the fastest run, in milliseconds."""
    best = float("inf")
    for _ in range(repeat):
        t0 = time.perf_counter()
        con.execute(sql, params).fetchall()
        best = min(best, time.perf_counter() - t0)
    return round(best * 1000, 1)


def db_size_mb():
    return round(os.path.getsize(WORK) / 1_048_576, 1)


print("fresh working copy:", WORK, "|", db_size_mb(), "MB")

fresh working copy: ride_knox_work.db | 40.1 MB


In [2]:
### PART A
    #Demonstrated rejections:
try_run("""
INSERT INTO stations (latitude)
VALUES ('Gay St')
""")
    # 'REFUSED -> NOT NULL constraint failed: stations.station_id'
try_run("""
INSERT INTO trips (start_station_id)
VALUES (S40)
""")
    # 'REFUSED -> no such column: S40'
try_run("""
INSERT INTO dock_changes (docks_after)
VALUES (0)
""")
    #'REFUSED -> NOT NULL constraint failed: dock_changes.station_id'

'REFUSED -> NOT NULL constraint failed: dock_changes.station_id'

In [3]:
### PART B

stations = pd.read_excel("stations_2026.xlsx") # Defines stations
trips25_original = pd.read_csv("trips_2025.csv") # Defines 2025 trips
trips26_original = pd.read_csv("trips_2026_h1.csv") # Defines 2026 trips 

stations.to_sql("stations_sql", con, if_exists="replace", index=False) # Loads stations
trips25 = pd.read_sql("""SELECT * FROM trips_2025""",con) # Loads trips25 (as a table, which was uploaded to DB Browser as a CSV)
trips26 = pd.read_sql("""SELECT * FROM trips_2026_h1""",con) # Loads trips26 (as a table, which was uplaoded to DB Browser as a CSV)

print(len(stations))
print(len(trips25))
print(len(trips26))
    # Checks the row count of each file

con.execute("BEGIN TRANSACTION;")
normalized_rider_types = con.execute("""
UPDATE trips_2026_h1
SET rider_type = CASE
            WHEN LOWER(TRIM(rider_type)) = 'day pass'
                THEN 'day_pass'
            ELSE LOWER(TRIM(rider_type))
        END
        WHERE rider_type <> LOWER(TRIM(rider_type))
           OR LOWER(TRIM(rider_type)) = 'day pass';
""")
    # Normalizes rider_type; used AI tool for IF THEN statement syntax

monthly26 = pd.read_sql("""
SELECT strftime('%m', start_time) AS month,
LOWER(rider_type) AS rider,
COUNT(*) AS trips
FROM trips_2026_h1
WHERE start_station_id <> 'S99'
AND (julianday(end_time) - julianday(start_time)) * 1440 BETWEEN 1 AND 1440
GROUP BY month, rider
ORDER BY month
""", con)
print(monthly26)
    # Confirms that rider types are normalized; also confirms that data appears through July, and that day_pass riders show up in month 3

25
250600
137341
   month     rider  trips
0     01    casual   3264
1     01    member   6410
2     02    casual   4185
3     02    member   8195
4     03    casual   6705
5     03  day_pass   1116
6     03    member  12382
7     04    casual   8918
8     04  day_pass   2251
9     04    member  15413
10    05    casual  10775
11    05  day_pass   3425
12    05    member  18048
13    06    casual  11365
14    06  day_pass   4125
15    06    member  19378


In [16]:
## PART C
trips26_07_original = pd.read_csv("trips_2026_07.csv") # Defines 2026_07 trips 
trips26_07 = pd.read_sql("""SELECT * FROM trips_2026_07""",con) # Loads trips26_07 (as a table, which was uploaded to DB Browser as a CSV)

monthly = pd.read_sql("""
SELECT strftime('%m', start_time) AS month,
LOWER(rider_type) AS rider,
COUNT(*) AS trips
FROM trips_2026_07
WHERE start_station_id <> 'S99'
AND (julianday(end_time) - julianday(start_time)) * 1440 BETWEEN 1 AND 1440
GROUP BY month, rider
ORDER BY month
""", con)
print(monthly)

count = pd.read_sql("""
SELECT
COUNT(*) AS all_trips,
COUNT(end_station_id) AS docked_trips,
ROUND(COUNT(end_station_id) * 100.0 / COUNT(*), 1) AS pct_docked
FROM trips_2026_07;
""", con)
print(count)

# For trips_2026_07.csv, noticed that ridership titles were inconsistent because some day pass riders were labeled as day_pass, while others were day pass or daypass. Noticed that not all trips were docked (2.6% were never docked). These could be stolen or otherwise unaccounted for. Noticed that while a majority of the trips were in July, 214 trips were in June.

  month     rider  trips
0    06    casual     60
1    06  day pass      2
2    06  day_pass      7
3    06    member    145
4    07    casual   7694
5    07  day pass   1366
6    07  day_pass   2820
7    07   daypass    738
8    07    member  19992
   all_trips  docked_trips  pct_docked
0      33085         32235        97.4


REFLECTION AND AI DISCLOSURE

REFLECTION:

AI Disclosure: I used a generative AI tool to explain how to save the work I did in SQL, as I was having trouble saving it as anything but a SQBPRO file. I also used it to help troubleshoot why I couldn't import the SOURCE, which is because it was saved in my folder called sql/. In part B, I was having trouble because rider_types could not be normalized with casing; instead, I saw that I needed to replace instances of spaces with an underscore, which I used AI for syntactical help.